In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator,TransformerMixin

In [ ]:
class IQROutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        
        q25 = X_df.quantile(0.25)
        q75 = X_df.quantile(0.75)
        iqr = q75 - q25
        
        for col in X_df.columns:
            self.lower_bounds_[col] = q25[col] - (self.factor * iqr[col])
            self.upper_bounds_[col] = q75[col] + (self.factor * iqr[col])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        
        return X_df.to_numpy() if isinstance(X, np.ndarray) else X_df

In [ ]:
housing =pd.read_csv("Housing.csv")

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(housing.drop(columns=['price']),housing['price'],test_size=0.25,random_state=42)

In [ ]:
t1=ColumnTransformer([
    ('impute_bathrooms',SimpleImputer(strategy='most_frequent'),[2]),
    ('impute_mainroad',SimpleImputer(strategy='most_frequent'),[4])
],remainder='passthrough')

In [ ]:
t2=ColumnTransformer([
    ('encode_mr_gr_bm_hw_ac_pa',OrdinalEncoder(categories=[['no','yes']]*6),[1,5,6,7,8,10]),
    ('encode_fr',OrdinalEncoder(categories=[['unfurnished','semi-furnished','furnished']]),[11])
],remainder='passthrough')

In [ ]:
t3=ColumnTransformer([
    ('outlier',IQROutlierCapper(),[8])
],remainder='passthrough')

In [ ]:
t5=LinearRegression()

In [ ]:
pipe=Pipeline([
    ('t1',t1),
    ('t2',t2),
    ('t3',t3),
    ('t5',t5)
])

In [ ]:
pipe.fit(x_train,y_train)

In [ ]:
y_pred=pipe.predict(x_test)

In [ ]:
from sklearn.metrics import r2_score ,mean_absolute_error,mean_squared_error
r2=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)

In [ ]:
(r2,mae,mse)

In [ ]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))